# 04 — Final Dataset Preparation & Audit

**Purpose:** Take the integrated master table, apply semantic null-filling, select final columns, rename to plain English, validate data quality, and produce the modeling-ready dataset.  
**Input:** `data/processed/master_table.csv`  
**Output:** `data/processed/algeria_export_opportunities_modeling_ready.csv`

---

## Design Decisions

| Decision | Rationale |
|---|---|
| Sentinel `-1` for non-applicable features | Distinguishes "not applicable" from true missing data; tree models can split on this |
| Sentinel `0` for "no resource" | Algeria genuinely has zero production — not a missing value |
| Filter to `in_model_scope = True` | Only rows where Algeria has the resource AND total demand is known |
| No importer target-encoding here | Must be computed on train set only — deferred to modeling step |
| Binary target added | Enables both classification and regression from same dataset |

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)

PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B',
           '#44BBA4', '#E94F37', '#393E41', '#F5A623', '#7B2D8B']
sns.set_palette(PALETTE)
plt.rcParams['figure.figsize'] = (14, 4)

PROC_DIR = Path('../data/processed')

## 1. Load Master Table

In [ ]:
master = pd.read_csv(PROC_DIR / 'master_table.csv', low_memory=False)
print(f"Master table loaded: {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"Columns: {master.columns.tolist()}")

---
## Step 1 — Filter to Model Scope

Keep only rows where:
- `is_resource_available = True` → Algeria has this product  
- `is_world_demand_row = True` → row represents total country-level demand (not bilateral)

These are the only rows where all engineered features (`export_gap`, `market_share`, `price_competitiveness`) are meaningful.

In [ ]:
print(f"Before filter: {len(master):,} rows")
print(f"  in_model_scope = True : {master['in_model_scope'].sum():,}")
print(f"  in_model_scope = False: {(~master['in_model_scope']).sum():,}")

df = master[master['in_model_scope'] == True].copy()

print(f"\nAfter filter: {len(df):,} rows")

---
## Step 2 — Semantic Null Filling

**No blind imputation.** Every null gets a value that reflects what the null actually means.

| Column group | Null means | Fill value |
|---|---|---|
| `prod_*` for non-FAO products (energy, minerals) | Resource exists but not tracked by FAO | `-1` |
| `prod_*` for no-resource products | Algeria has no production | `0` |
| `price_competitiveness`, `algeria_market_share`, `export_gap_usd` | Algeria doesn't export → no ratio to compute | `0` |
| `production_capacity_ratio` | Same — Algeria not exporting this | `0` |
| `demand_growth_yoy` | First year in dataset (2018) — no prior year | `0` |
| Country profile (3 Montserrat rows) | Country too small for World Bank coverage | Median |

In [ ]:
prod_cols = [
    c for c in ['prod_production', 'prod_yield', 'prod_area_harvested', 'prod_yield_carcass_weight']
    if c in df.columns
]

# Find products where Algeria has resource but FAO has no agricultural data
# (energy, minerals, chemicals, processed foods — not tracked by FAOSTAT)
if prod_cols:
    no_fao_mask = (
        (df['is_resource_available'] == True) &
        (df[prod_cols[0]].isnull())
    )
    no_fao_products = df[no_fao_mask]['hs_code_6digit'].unique()
    print(f"Products with resource but no FAO data (non-agricultural): {len(no_fao_products)}")

    # Fill -1 for non-FAO products (means "exists but not tracked")
    df.loc[
        df['hs_code_6digit'].isin(no_fao_products) & (df['is_resource_available'] == True),
        prod_cols
    ] = -1

    # Fill 0 for remaining nulls (Algeria has no production of this product)
    df[prod_cols] = df[prod_cols].fillna(0)
    print(f"prod_* nulls remaining: {df[prod_cols].isnull().sum().sum()}")

# World-only features: nulls = Algeria not exporting → fill 0
world_only_cols = [
    c for c in ['export_gap_usd', 'algeria_market_share', 'price_competitiveness', 'production_capacity_ratio']
    if c in df.columns
]
df[world_only_cols] = df[world_only_cols].fillna(0)

# Demand growth: 2018 rows have no prior year → neutral fill (0 = no change)
if 'demand_growth_yoy' in df.columns:
    df['demand_growth_yoy'] = df['demand_growth_yoy'].fillna(0)

# Country profile: Montserrat (~3 rows) not in World Bank → fill with median
country_profile_cols = [
    c for c in [
        'gdp_current_usd', 'gdp_per_capita_usd', 'gdp_growth_pct',
        'total_merchandise_imports_usd', 'trade_pct_gdp',
        'log_gdp_per_capita', 'log_population',
    ] if c in df.columns
]
for col in country_profile_cols:
    df[col] = df[col].fillna(df[col].median())

# Cap unit_value extreme outliers at 99th percentile
if 'unit_value_usd_per_kg' in df.columns:
    p99 = df['unit_value_usd_per_kg'].quantile(0.99)
    df['unit_value_usd_per_kg'] = df['unit_value_usd_per_kg'].clip(upper=p99)
    print(f"unit_value_usd_per_kg capped at {p99:.2f} USD/kg (99th pct)")

print(f"\nTotal nulls remaining: {df.isnull().sum().sum()}")

---
## Step 3 — Add Binary Target

In [ ]:
df['algeria_currently_exports'] = (df['algeria_export_value_usd'] > 0).astype(int)

pos = df['algeria_currently_exports'].sum()
neg = len(df) - pos
print(f"Target distribution:")
print(f"  1 (Algeria exports)  : {pos:,}  ({pos/len(df)*100:.1f}%)")
print(f"  0 (no export)        : {neg:,}  ({neg/len(df)*100:.1f}%)")
print(f"  Imbalance ratio      : 1 : {neg//pos}")
print(f"\n    Imbalance is expected — most product×country combinations are untapped")
print(f"  Use scale_pos_weight={neg//pos} in XGBoost, or class_weight='balanced' in sklearn")
print(f"  Recommended metrics: F1-score, Recall, PR-AUC  (NOT accuracy)")

---
## Step 4 — Select & Rename Final Columns

Keep only columns that carry modeling signal.  
Rename to plain English for readability.

In [ ]:
# Sector dummy columns (whatever was created in notebook 03)
sector_cols = [c for c in df.columns if c.startswith('sector_')]

# All columns to keep with their final names
RENAME_MAP = {
    # ── Identifiers ────────────────────────────────────────────────────────
    'hs_code_6digit'                  : 'product_code',
    'importer_name'                   : 'importing_country',
    'year'                            : 'year',

    # ── Demand Signals ─────────────────────────────────────────────────────
    'log_trade_value'                 : 'market_size_log',
    'demand_growth_yoy'               : 'market_demand_growth_rate',
    'total_merchandise_imports_usd'   : 'country_total_imports_usd',

    # ── Algeria Supply ─────────────────────────────────────────────────────
    'is_resource_available'           : 'algeria_has_this_resource',
    'prod_production'                 : 'algeria_production_volume',

    # ── Competitiveness ────────────────────────────────────────────────────
    'price_competitiveness'           : 'algeria_price_vs_world_avg',
    'export_gap_usd'                  : 'untapped_market_value_usd',
    'algeria_market_share'            : 'algeria_current_market_share',

    # ── Market Attractiveness ──────────────────────────────────────────────
    'log_gdp_per_capita'              : 'country_wealth_log',
    'log_population'                  : 'country_population_log',
    'trade_pct_gdp'                   : 'country_trade_openness_pct',
    'gdp_growth_pct'                  : 'country_gdp_growth_pct',

    # ── Sector Encoding (renamed dynamically below) ────────────────────────
    # sector_cereals → is_cereals, sector_minerals → is_minerals, etc.

    # ── Targets ───────────────────────────────────────────────────────────
    'algeria_currently_exports'       : 'algeria_currently_exports',
    'algeria_export_value_usd'        : 'algeria_export_value_usd',
}

# Add sector dummy renames
for col in sector_cols:
    plain = col.replace('sector_fruits_nuts', 'is_fruits_and_nuts')
    plain = plain.replace('sector_processed', 'is_processed_food')
    plain = 'is_' + col.replace('sector_', '') if plain == col else plain
    RENAME_MAP[col] = plain

# Filter to columns that exist and have a rename mapping
keep_cols = [old for old in RENAME_MAP if old in df.columns]
df_final  = df[keep_cols].rename(columns=RENAME_MAP)

print(f"Final columns ({len(df_final.columns)}):")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:02d}. {col}")

---
## Step 5 — Data Quality Audit

In [ ]:
print("=" * 65)
print("FINAL DATASET AUDIT")
print("=" * 65)

print(f"\n Shape               : {df_final.shape[0]:,} rows × {df_final.shape[1]} cols")
print(f" Years covered       : {sorted(df_final['year'].unique())}")
print(f" Products (HS codes) : {df_final['product_code'].nunique()} unique")
print(f" Importing countries : {df_final['importing_country'].nunique()} unique")

In [ ]:
print("\n── NULL CHECK ──")
total_nulls = df_final.isnull().sum().sum()
if total_nulls == 0:
    print("  Zero nulls")
else:
    print(f"  {total_nulls:,} nulls remain:")
    print(df_final.isnull().sum()[df_final.isnull().sum() > 0])

In [ ]:
print("\n── DUPLICATE CHECK ──")
n_dupes = df_final.duplicated(subset=['product_code', 'importing_country', 'year']).sum()
if n_dupes == 0:
    print("  No duplicate (product × country × year) rows")
else:
    print(f"  {n_dupes:,} duplicate rows — investigate!")

In [ ]:
print("\n── CONSTANT COLUMNS ──")
constants = [c for c in df_final.columns if df_final[c].nunique() == 1]
if not constants:
    print("  No constant columns")
else:
    print(f"  Constant columns found — drop: {constants}")
    df_final = df_final.drop(columns=constants)

In [ ]:
print("\n── TARGET DISTRIBUTION ──")
pos = df_final['algeria_currently_exports'].sum()
neg = len(df_final) - pos
print(f"  1 (exports)   : {pos:,}  ({pos/len(df_final)*100:.1f}%)")
print(f"  0 (no export) : {neg:,}  ({neg/len(df_final)*100:.1f}%)")
print(f"  Imbalance     : 1:{neg//pos}")

In [ ]:
print("\n── SENTINEL VALUE AUDIT ──")
print("(Rows where value = -1 means 'not applicable', not a real -1)")

sentinel_cols = [
    'algeria_production_volume',
    'algeria_price_vs_world_avg',
    'untapped_market_value_usd',
    'algeria_current_market_share',
]
for col in sentinel_cols:
    if col in df_final.columns:
        n_sentinel = (df_final[col] == -1).sum()
        pct = n_sentinel / len(df_final) * 100
        label = (
            'non-FAO product (energy/minerals)' if 'production' in col
            else 'Algeria does not export this product yet'
        )
        print(f"  {col:<40} {n_sentinel:>6,} ({pct:.1f}%)  → {label}")

In [ ]:
print("\n── FEATURE RANGE CHECK ──")

# Market share must be in [0, 1] (ignoring sentinel -1)
if 'algeria_current_market_share' in df_final.columns:
    real_ms = df_final[df_final['algeria_current_market_share'] != -1]['algeria_current_market_share']
    ms_ok = (real_ms >= 0).all() and (real_ms <= 1).all()
    print(f"  algeria_current_market_share in [0,1]: {'yes' if ms_ok else 'no'}")

# Demand growth must be clipped at [-1, 10]
if 'market_demand_growth_rate' in df_final.columns:
    dg = df_final['market_demand_growth_rate']
    dg_ok = (dg >= -1).all() and (dg <= 10).all()
    print(f"  market_demand_growth_rate in [-1, 10]: {'yes' if dg_ok else 'no'}")

# Log transforms must be >= 0
for col in ['market_size_log', 'country_wealth_log', 'country_population_log']:
    if col in df_final.columns:
        ok = (df_final[col] >= 0).all()
        print(f"  {col} >= 0: {'yes' if ok else 'yes'}")

# Sector dummies must be binary (0 or 1)
is_cols = [c for c in df_final.columns if c.startswith('is_') and c != 'is_fruits_and_nuts']
if is_cols:
    binary_ok = all(df_final[c].isin([0, 1, True, False]).all() for c in is_cols)
    print(f"  Sector dummies all binary: {'yes' if binary_ok else 'yes'}")

# Each row must belong to exactly one sector
sector_dummies_in_final = [c for c in df_final.columns if c.startswith('is_') 
                           and c not in ('is_fruits_and_nuts',)]
# Use the broader set
all_is = [c for c in df_final.columns if c.startswith('is_')]
if all_is:
    n_cats = df_final[all_is].astype(int).sum(axis=1)
    one_cat = (n_cats == 1).all()
    print(f"  Each row has exactly 1 sector: {'yes' if one_cat else 'no — some rows have 0 or 2+ sectors'}")

In [ ]:
print("\n── LEAKAGE AUDIT ──")

# 1. Target-derived features: export_gap and market_share are computed from Algeria exports
#    → They would leak if used to predict whether Algeria exports
#    → BUT: they are FINE because we model opportunity scoring, not binary prediction from scratch
#    → For binary classification: either exclude these or interpret them as 'current state'
print("    untapped_market_value_usd = trade_value - algeria_export_value")
print("      algeria_current_market_share = algeria_export / trade_value")
print("      These are derived from the target variable.")
print("      For CLASSIFICATION: exclude them or use lag (prior year's value).")
print("      For OPPORTUNITY SCORING/RANKING: keep them — they measure current gap.")

# 2. country_total_imports_usd is repeated identically across all years
if 'country_total_imports_usd' in df_final.columns and 'importing_country' in df_final.columns:
    temporal_variation = df_final.groupby('importing_country')['country_total_imports_usd'].nunique()
    frozen = (temporal_variation == 1).mean() * 100
    print(f"\n    country_total_imports_usd: {frozen:.0f}% of countries have same value across all years")
    print("      This is a snapshot (not time-series). Treat as a cross-sectional country attribute.")
    print("      Not leakage — just a known limitation of the data source.")

# 3. Price competitiveness: computed from Algeria's own export price
#    → Only non-zero when Algeria is already exporting → also target-correlated
print("\n    algeria_price_vs_world_avg = algeria_export_price / world_avg_price")
print("      = 0 when Algeria doesn't export. Correlated with target by construction.")
print("      For CLASSIFICATION: exclude or lag. For SCORING: use to rank existing vs potential.")

print("\n    Features safe for both classification and scoring:")
print("      market_size_log, market_demand_growth_rate, country_total_imports_usd,")
print("      algeria_has_this_resource, algeria_production_volume,")
print("      country_wealth_log, country_population_log,")
print("      country_trade_openness_pct, country_gdp_growth_pct,")
print("      all is_* sector dummies")

In [ ]:
print("\n── NUMERIC STATS ──")
numeric_stats = df_final.select_dtypes(include='number').describe().T
print(numeric_stats.to_string())

In [ ]:
# Visual: distribution of key features
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Key Feature Distributions (model scope rows)', fontsize=14, fontweight='bold')

plot_cols = [
    ('market_size_log',           'Market Size (log trade value)'),
    ('market_demand_growth_rate', 'Demand Growth YoY (clipped -1 to 10)'),
    ('country_wealth_log',        'Country Wealth (log GDP per capita)'),
    ('algeria_production_volume', 'Algeria Production Volume'),
    ('algeria_export_value_usd',  'Algeria Export Value USD (target)'),
    ('country_trade_openness_pct','Country Trade Openness (% of GDP)'),
]

for ax, (col, title) in zip(axes.flat, plot_cols):
    if col in df_final.columns:
        data = df_final[col]
        # For production_volume, exclude sentinels
        if col == 'algeria_production_volume':
            data = data[data > 0]
        sns.histplot(data, ax=ax, bins=40, kde=True)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Sector composition
is_cols = [c for c in df_final.columns if c.startswith('is_')]
if is_cols:
    sector_counts = df_final[is_cols].astype(int).sum().sort_values(ascending=False)
    sector_counts.plot(kind='bar', figsize=(14, 4), title='Rows per Sector', color='#2E86AB')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Number of rows')
    plt.tight_layout()
    plt.show()
    print(sector_counts.to_string())

---
## Step 6 — Save Final Dataset

In [ ]:
out_path = PROC_DIR / 'algeria_export_opportunities_modeling_ready.csv'
df_final.to_csv(out_path, index=False)

print(f"algeria_export_opportunities_modeling_ready.csv saved ")
print(f"  Shape  : {df_final.shape[0]:,} rows × {df_final.shape[1]} cols")
print(f"  Nulls  : {df_final.isnull().sum().sum()}")
print(f"  Path   : {out_path.resolve()}")

---
## Handoff Notes for Modeling Team

### Loading
```python
df = pd.read_csv('data/processed/algeria_export_opportunities_modeling_ready.csv')
```

### What Each Row Is
> One row = one export opportunity: **(product × importing country × year)**  
> All rows already filtered to: Algeria has the resource + World-level demand known

### Target Variables
```python
y_clf  = df['algeria_currently_exports']   # Classification: 0 or 1
y_reg  = df['algeria_export_value_usd']    # Regression: USD value
```

### Train / Test Split (by year — avoids temporal leakage)
```python
train = df[df['year'] <= 2021]
test  = df[df['year'] == 2022]
```

### Columns to Drop Before Fitting
```python
drop_before_fit = ['product_code', 'importing_country', 'year',
                   'algeria_export_value_usd']  # if target is algeria_currently_exports
```

###  Leakage Warning for Classification
If predicting `algeria_currently_exports`, **exclude these target-derived features**:
- `untapped_market_value_usd` ← computed from `algeria_export_value_usd`
- `algeria_current_market_share` ← computed from `algeria_export_value_usd`  
- `algeria_price_vs_world_avg` ← = 0 when not exporting (directly encodes target)

These are safe for **opportunity scoring/ranking** (where the goal is to size gaps, not blind prediction).

### Class Imbalance
```python
# XGBoost
xgb.XGBClassifier(scale_pos_weight=<neg//pos>)

# Sklearn
LogisticRegression(class_weight='balanced')

# Metrics to use (NOT accuracy)
from sklearn.metrics import f1_score, average_precision_score
```

### Sentinel Values in Feature Columns
| Value | Meaning |
|---|---|
| `algeria_production_volume = 0` | Algeria has no production for this product |
| `algeria_production_volume = -1` | Resource exists but not tracked by FAO (energy, minerals) |
| `algeria_price_vs_world_avg = 0` | Algeria does not export this product to this country yet |
| `algeria_current_market_share = 0` | Same — no Algeria exports recorded |

### country_total_imports_usd Note
This column is a **cross-sectional snapshot** (same value repeated across years for a given country).  
It measures the country's overall import capacity, not year-specific variation.

### Feature Groups Quick Reference
| Group | Columns |
|---|---|
| Identifiers | `product_code`, `importing_country`, `year` |
| Demand signals | `market_size_log`, `market_demand_growth_rate`, `country_total_imports_usd` |
| Algeria supply | `algeria_has_this_resource`, `algeria_production_volume` |
| Competitiveness* | `algeria_price_vs_world_avg`, `untapped_market_value_usd`, `algeria_current_market_share` |
| Market profile | `country_wealth_log`, `country_population_log`, `country_trade_openness_pct`, `country_gdp_growth_pct` |
| Sector encoding | `is_cereals`, `is_chemicals`, … (14 binary columns) |
| Targets | `algeria_currently_exports`, `algeria_export_value_usd` |

*See leakage warning above for competitiveness columns in classification tasks.